In [0]:
from pyspark.sql.session import SparkSession
spark= SparkSession.builder.appName("test").getOrCreate()

In [0]:
df=spark.read.csv("/FileStore/employees_with_nulls_duplicates.csv",header=True,inferSchema=True)

In [0]:
df=spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema",True)\
    .option("sep",",")\
    .load("/FileStore/employees_with_nulls_duplicates.csv")

INFO:py4j.clientserver:Received command c on object id p0
DEBUG:py4j.clientserver:Command to send: c
t
getDbutils
e

DEBUG:py4j.clientserver:Answer received: !yro2549
DEBUG:py4j.clientserver:Command to send: c
o2549
notebook
e

DEBUG:py4j.clientserver:Answer received: !yro2550
DEBUG:py4j.clientserver:Command to send: c
o2550
getContext
e

DEBUG:py4j.clientserver:Answer received: !yro2551
DEBUG:py4j.clientserver:Command to send: c
o2551
apiUrl
e

DEBUG:py4j.clientserver:Answer received: !yro2552
DEBUG:py4j.clientserver:Command to send: c
o2552
isDefined
e

DEBUG:py4j.clientserver:Answer received: !ybtrue
DEBUG:py4j.clientserver:Command to send: c
o2552
get
e

DEBUG:py4j.clientserver:Answer received: !yshttps://community.cloud.databricks.com
DEBUG:py4j.clientserver:Command to send: c
o2551
apiToken
e

DEBUG:py4j.clientserver:Answer received: !yro2553
DEBUG:py4j.clientserver:Command to send: c
o2553
isDefined
e

DEBUG:py4j.clientserver:Answer received: !ybtrue
DEBUG:py4j.clientserver:Comm

In [0]:
# df.select(mean(df.age)).collect()[0][0]
# df.fillna( df.select(median(col("id"))).collect()[0][0])
# df.fillna( df.select(mean(df.salary)).collect()[0][0])
# df.fillna( df.select(mode(df.region)).collect()[0][0])
# df.fillna( df.select(mode(df.country)).collect()[0][0])
# df.fillna( df.select(mode(df.name)).collect()[0][0])
# df.fillna( df.select(mode(df.gender)).collect()[0][0])
# df.fillna( df.select(mode(df.city)).collect()[0][0])

df1=df.fillna({"id":df.select(median(col("id"))).collect()[0][0],\
    "salary":df.select(mean(df.salary)).collect()[0][0],\
    "region":df.select(mode(df.region)).collect()[0][0],\
    "country":df.select(mode(df.country)).collect()[0][0],\
    "name":df.select(mode(df.name)).collect()[0][0],\
    "gender":df.select(mode(df.gender)).collect()[0][0],\
    "city":df.select(mode(df.city)).collect()[0][0]})
    
df.dtypes

Out[5]: [('id', 'double'),
 ('name', 'string'),
 ('region', 'string'),
 ('country', 'string'),
 ('salary', 'double'),
 ('gender', 'string'),
 ('city', 'string'),
 ('age', 'double'),
 ('deptname', 'string')]

In [0]:
from pyspark.sql.functions import col,mean,mode,median,trim,lit,asc,desc,dense_rank,rank,row_number,when,expr,count
from pyspark.sql.window import Window
# col=df.columns()
# df.filter(col().isNull()).display()

In [0]:
for i in df.columns:
    if df.select(i).dtypes[0][1]=='double':
        df.select(i).fillna({ i:df.select(median(col(i))).collect()[0][0]})
        print(i)

df.display()

id
salary
age


id,name,region,country,salary,gender,city,age,deptname
73.0,Ivan,North,Canada,57364.0,Male,Paris,60.0,Sales
83.0,Diana,South,USA,90067.0,null,London,59.0,Engineering
19.0,Judy,South,UK,91791.0,Female,Paris,26.0,HR
80.0,Alice,West,UK,70929.0,Female,Berlin,53.0,Sales
2.0,Ivan,South,USA,null,Female,New York,31.0,null
89.0,Diana,North,UK,82142.0,null,Paris,49.0,null
33.0,Charlie,West,Germany,null,Female,London,26.0,HR
22.0,Diana,South,null,89431.0,Female,Berlin,51.0,Marketing
41.0,Judy,null,null,107541.0,Male,Paris,null,Sales
42.0,Charlie,South,UK,66654.0,Female,New York,22.0,HR


In [0]:
import logging
from pyspark.sql.functions import col,mean,mode,median,trim,asc,desc,dense_rank,rank,row_number,when,expr,count
from pyspark.sql.window import Window
from pyspark.sql.session import SparkSession

logging.basicConfig(filename="\FileStore\logging.log",level=10)

def read_csv(spark,path):
    try:
        df=spark.read.format("csv")\
            .option("header",True)\
            .option("inferSchema",True)\
            .option("sep",",")\
            .load(path)
        logging.info("csv file read successfully!!")
        return df
    except Exception as e:
        logging.error(f"failed to read file {e}")

def drop_na(df):
    try:
        df=df.na.drop(subset=['id','name'])
        logging.info("remove  null from id name successfully done!!")
        return df
    except Exception as e:
        logging.error("failed to remove null")

def fillna_df(df):
    try:
        fill_dict={}
        for col_name in df.columns:
            if df.select(col_name).dtypes[0][1] in ['float','double']:
                fill_dict[col_name]=float(df.select(median(col(col_name))).collect()[0][0])
            elif df.select(col_name).dtypes[0][1]=='int':
                fill_dict[col_name]=int(df.select(mean(col(col_name))).collect()[0][0])
            elif df.select(col_name).dtypes[0][1]=='string':
                fill_dict[col_name]=df.select(mode(col(col_name))).collect()[0][0]
        df=df.fillna(fill_dict)
        logging.info("fill null values successfully")
        return df
    except Exception as e:
        logging.error("failed to fill na values")

def trim_column(df):
    try:
        d=[trim(col(i)) for i in df.columns]
        df1 = df.select(*d)
        li=[column[column.find("(")+1 :len(column)] for column in df.columns]
        del df
        df=df1.toDF(*li)
        logging.info("trim column  successfully")
        return df
    except Exception as e:
        logging.error("trim column failed successfully")

def drop_duplicate(df):
    try:
        df= df.dropDuplicates(subset=['id','name'])
        logging.info("drop duplicates successfully")
        return df
    except Exception as e:
        logging.error("failed drop duplicates")

def ranking(df):
    try:
        w=Window.partitionBy("city","deptname").orderBy(col("salary").desc())
        df=df.withColumn("raw_number",row_number().over(w)).withColumn("rank",rank().over(w)).withColumn("dense_rank",dense_rank().over(w))
        logging.info("ranking succrssfully done")
        return df
    except Exception as e:
        logging.error("failed to ranking")

def encode_df(df):
    try:
        df= df.withColumn("gen_encode",expr("case when  gender=='Male' then 1 when gender =='Female' then 2 else 0 end ")).withColumn("city_encode",expr(" case when city=='Berlin' then 100 when city=='London' then 200 when city=='Paris' then 300 when city =='Toronto' then 400 when city=='New York' then 500 else 0 end"))
        logging.info("ecode successfully done")
        return df
    except Exception as e:
        logging.error("encode failed",e)


def city_count(df):
    try:
        
        df=df.groupBy("city").agg(count("city"))
        logging.info("count of city done")
        return df
    except Exception as e:
        logging.error("successfully done ")
    
def main():
    spark = SparkSession.builder.appName("test").getOrCreate()
    df=read_csv(spark,"/FileStore/employees_with_nulls_duplicates.csv")
    df = drop_na(df)
    df = fillna_df(df)
    df = trim_column(df)
    df = drop_duplicate(df)
    df = ranking(df)
    df = encode_df(df)
    df_city_count = city_count(df) 
    df.display()
    df_city_count.display()

if '__main__'==__name__:
    main()



id,name,region,country,salary,gender,city,age,deptname,raw_number,rank,dense_rank,gen_encode,city_encode
13.0,Bob,South,Germany,95281.0,Female,Berlin,27.0,Engineering,1,1,1,2,100
17.0,Ivan,West,USA,81962.0,Male,Berlin,53.0,Finance,1,1,1,1,100
5.0,Judy,West,USA,79732.0,Female,Berlin,60.0,Finance,2,2,2,2,100
61.0,Judy,West,Germany,78428.0,Male,Berlin,43.0,Finance,3,3,3,1,100
20.0,Ivan,North,USA,75407.0,Female,Berlin,29.0,Finance,4,4,4,2,100
37.0,Heidi,West,France,69925.0,Male,Berlin,26.0,HR,1,1,1,1,100
22.0,Diana,South,Germany,89431.0,Female,Berlin,51.0,Marketing,1,1,1,2,100
75.0,Alice,North,Germany,81837.0,Male,Berlin,37.0,Sales,1,1,1,1,100
87.0,Diana,East,Germany,79732.0,Female,Berlin,38.0,Sales,2,2,2,2,100
43.0,Frank,West,Germany,71959.0,Male,Berlin,44.0,Sales,3,3,3,1,100


city,count(city)
Berlin,14
London,9
Paris,16
Toronto,26
New York,14


In [0]:


# logger = logging.getLogger('my_logger')
# logger.setLevel(logging.DEBUG)

# if not logger.handlers:
#     handler = logging.StreamHandler()
#     formatter = logging.Formatter('%(asctime)s %(levelname)s: %(message)s')
#     handler.setFormatter(formatter)
#     logger.addHandler(handler)

#for file
# dbfs_path = "/dbfs/tmp/app22.log"
# logger = logging.getLogger('my_logger')
# logger.setLevel(logging.DEBUG)
# if not logger.handlers:
#     handler = logging.FileHandler(dbfs_path)
#     formatter = logging.Formatter('%(asctime)s %(levelname)s: %(message)s')
#     handler.setFormatter(formatter)
#     logger.addHandler(handler)
# # df.select("age").dtypes#[0][1]
# df=df.na.drop(subset=['id','name'])
# fill_dict={}
# s=df.dtypes
# a,b=zip(*s)
# print(set(a))
# print(s)
# for j in set(a):
#     if df.select(j).dtypes[0][1] in ['float','double']:
#         fill_dict[j]=float(df.select(median(col(j))).collect()[0][0])
#     elif df.select(j).dtypes[0][1]=='int':
#         fill_dict[j]=int(df.select(mean(col(j))).collect()[0][0])
#     elif df.select(j).dtypes[0][1]=='string':
#         fill_dict[j]=df.select(mode(col(j))).collect()[0][0]
# print(fill_dict)
# df.fillna(fill_dict)#.display()
# d=[trim(col(i)) for i in df.columns]
# df.select(*d)#.display()
# df.dropDuplicates(subset=['id','name'])
# # df=df.select(col())
# li=[column[column.find("("):len(column)-1] for column in df.columns]
# print(li)
# df.select("city").distinct().show()
# w=Window.partitionBy("city").orderBy(col("salary").desc())
# df.withColumn("raw_number",row_number().over(w)).withColumn("rank",rank().over(w)).withColumn("dense_rank",dense_rank().over(w))#.display()
# df.withColumn("gen_encode",when (col('("case when  gender=='Male' then 1 when gender =='Female' then 2 else 0 end ")).withColumn("city_encode",expr(" case when city=='Berlin' then 100 when city=='London' then 200 when city=='Paris' then 300 when city=='Toronto' then 400 when city=='New York' then 500 else 0 end"))df.select(mean(df.age)).collect()[0][0]

Out[35]: 38.44565217391305

In [0]:
s=df.dtypes
a,b=zip(*s)
print(a)
print(set(b))
print(s)

('id', 'name', 'region', 'country', 'salary', 'gender', 'city', 'age', 'deptname')
{'double', 'string'}
[('id', 'double'), ('name', 'string'), ('region', 'string'), ('country', 'string'), ('salary', 'double'), ('gender', 'string'), ('city', 'string'), ('age', 'double'), ('deptname', 'string')]


In [0]:
# df1 = df.trim(select(trim(col(a[0])),trim(col(a[1])),trim(col(a[2])),trim(col(a[3])),trim(col(a[4])),trim(col(a[5])),trim(col(a[6])),trim(col(a[7])),trim(col(a[8])))).display()
# d=[trim(col(i)) for i in df.columns]
# df.select(*d).columns#display()
# df1.columns
# df1.display()
d=[trim(col(i)) for i in df.columns]
df1 = df.select(*d)
li=[column[column.find("(")+1 :len(column)] for column in df.columns]
df1.toDF(*li).columns


Out[9]: ['id',
 'name',
 'region',
 'country',
 'salary',
 'gender',
 'city',
 'age',
 'deptname']

In [0]:
# df.select(*df.columns).display()
# trim(df.name).display()
# df.select(mode(col(j)))[0][0]
# df.columns
# li=[column[column.find("("):len(column)-1] for column in df.columns]
# print(li)

'jhzbh()'['jhzbh()'.find("()")-1:len('jhzbh()')-1]

Out[17]: 'h('

In [0]:
 d=[trim(col(i)) for i in df.columns]
 df.select(*d)

Out[56]: DataFrame[trim(id): string, trim(name): string, trim(region): string, trim(country): string, trim(salary): string, trim(gender): string, trim(city): string, trim(age): string, trim(deptname): string]

In [0]:
'trim(id)'.strip("trm()")

Out[59]: 'im(id'

In [0]:
   df.select("city").distinct().show()

+--------+
|    city|
+--------+
|  Berlin|
|    null|
|  London|
|   Paris|
| Toronto|
|New York|
+--------+



In [0]:
df.groupBy("city").agg(count("city")).show()

+--------+-----------+
|    city|count(city)|
+--------+-----------+
|  Berlin|         16|
|    null|          0|
|  London|         10|
|   Paris|         19|
| Toronto|         20|
|New York|         16|
+--------+-----------+



In [0]:
# for column in df1.columns:
#     print(column[column.find("(")+1:len(column)-1])
# df
li=[column[column.find("(")+1 :len(column)-1] for column in df1.columns]
df.toDF(*li).display()

id,name,region,country,salary,gender,city,age,deptname
73.0,Ivan,North,Canada,57364.0,Male,Paris,60.0,Sales
83.0,Diana,South,USA,90067.0,null,London,59.0,Engineering
19.0,Judy,South,UK,91791.0,Female,Paris,26.0,HR
80.0,Alice,West,UK,70929.0,Female,Berlin,53.0,Sales
2.0,Ivan,South,USA,null,Female,New York,31.0,null
89.0,Diana,North,UK,82142.0,null,Paris,49.0,null
33.0,Charlie,West,Germany,null,Female,London,26.0,HR
22.0,Diana,South,null,89431.0,Female,Berlin,51.0,Marketing
41.0,Judy,null,null,107541.0,Male,Paris,null,Sales
42.0,Charlie,South,UK,66654.0,Female,New York,22.0,HR
